In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys

import torch

from spice import SpiceEstimator

sys.path.append("../../..")
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, get_dataset, generate_behavior
import spice_castro2025, spice_castro2025_2

## Load dataset

In [2]:
path_data = 'data/eckstein2024.csv'
test_sessions = (2,)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([4158, 150, 1, 13])
Number of participants: 862
Number of actions in dataset: 4


In [3]:
# keep only 100 participants for rapid prototyping
from spice import SpiceDataset

keep_participants = torch.arange(0, 100)

mask_train = torch.isin(dataset_train.xs[..., -1], keep_participants)
mask_test = torch.isin(dataset_test.xs[..., -1], keep_participants)

xs, ys = dataset_train.xs[mask_train], dataset_train.ys[mask_train]
dataset_train = SpiceDataset(xs, ys)

xs, ys = dataset_test.xs[mask_test], dataset_test.ys[mask_test]
dataset_test = SpiceDataset(xs, ys)


## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
# fitting without SINDy coefficients
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025_2.SpiceModel,
        spice_config=spice_castro2025_2.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1,
        
        sindy_weight=0,
        sindy_library_polynomial_degree=2,
        sindy_pruning_frequency=100,
        sindy_ensemble_pruning=0.05,
        sindy_threshold_pruning=0.01,
        sindy_alpha=0.0001,
        
        save_path_spice=path_spice,
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [11]:
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025_2.SpiceModel,
        spice_config=spice_castro2025_2.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1000,
        warmup_steps=200,
        learning_rate=0.01,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,

        verbose=True,
        # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [12]:
estimator.load_spice(path_spice)
estimator.aggregate_coefficients()

In [13]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)


Example SPICE model (participant 0):
value_reward_env[t+1]        = 1.388 value_reward_env[t] 
value_reward_chosen[t+1]     = -0.455 1 + 0.518 value_reward_chosen[t] + 0.504 reward[t] + -0.549 reward_env^2 + 0.524 reward[t]^2 
value_reward_not_chosen[t+1] = 1.0 value_reward_not_chosen[t] 
value_choice_chosen[t+1]     = 0.115 1 + 0.699 value_choice_chosen[t] + 0.149 value_choice_chosen^2 
value_choice_not_chosen[t+1] = -0.055 1 + 0.655 value_choice_not_chosen[t] 
volatility_chosen[t+1]       = 0.077 1 + 0.684 volatility_chosen[t] + 1.156 dvalue 
volatility_not_chosen[t+1]   = 1.0 volatility_not_chosen[t] 


## Benchmarking

### Castro2025 benchmark model

In [5]:
# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=dataset_train.n_participants,
    n_actions=dataset_train.n_actions,
    batch_first=True,
    )

path_cfs = path_spice.replace('spice_', 'cfs_')

In [ ]:
optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
epochs = 1000

cfs = training(
    model=cfs,
    optimizer=optimizer_cfs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
)

torch.save(cfs.state_dict(), path_cfs)

In [6]:
cfs.load_state_dict(torch.load(path_cfs, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [7]:
gru = GRUModel(
    n_actions=dataset_train.n_actions, 
    additional_inputs=2, 
    dropout=0.1,
    hidden_size=32,
    )
path_gru = path_spice.replace('spice_', 'gru_')

In [ ]:
epochs = 1000
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer_gru,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

In [8]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [ ]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from weinhardt2026.analysis.analysis_value_trajectories import plot_value_trajectories, plot_value_trajectories_multi
from analysis_generative import analysis_generative_behavior

## General Analysis

In [15]:
# Dataset-specific behavioral analysis placeholder.
# Replace with Eckstein2024-specific columns if needed.

In [16]:
print("Fitted Castro2025 parameters:")
print("\nBeta_r")
print(cfs.beta_r)
print("\nLapse")
print(cfs.lapse)
print("\nPrior")
print(cfs.prior)
print("\nAlpha exploration rate")
print(cfs.alpha_exploration_rate)
print("\nDecay rate")
print(cfs.decay_rate)
print("\nGamma")
print(cfs.gamma)
print("\nTemperature")
print(cfs.temperature)

Fitted Castro2025 parameters:

Beta_r
tensor([2.6588, 2.5872, 2.4406, 2.6166, 1.5054, 2.2513, 2.1100, 2.1039, 1.3746,
        2.0745, 2.6667, 2.1328, 2.3358, 2.6025, 2.1986, 2.1311, 2.8331, 2.4313,
        2.6382, 3.0562, 2.2559, 2.3994, 1.6902, 2.9913, 1.8730, 1.9825, 1.6819,
        2.4229, 2.0172, 2.3439, 1.9366, 2.0212, 2.0829, 2.6930, 2.3627, 2.4913,
        2.2294, 2.5848, 2.2695, 1.6097, 2.0131, 1.8442, 2.4099, 1.7877, 2.4884,
        2.5506, 2.2508, 1.9150, 2.4284, 2.4954, 2.2797, 1.9747, 2.2009, 1.3223,
        1.7321, 1.8294, 1.8130, 2.4777, 2.4677, 2.4375, 1.7650, 2.0791, 2.4099,
        1.9940, 2.5261, 1.7374, 2.3255, 2.0326, 1.4980, 1.6791, 1.8890, 1.9227,
        3.0623, 3.0400, 1.6810, 2.5141, 2.1057, 1.5235, 2.4788, 3.1193, 2.7911,
        2.1080, 2.4988, 1.9093, 1.6636, 2.5500, 1.7713, 2.7464, 1.7287, 2.7249,
        1.5610, 2.2452, 2.2940, 3.5569, 1.9799, 1.0523, 2.3127, 1.8893, 2.3050,
        2.3087, 2.0803, 1.8150, 2.6862, 2.3649, 2.2447, 2.5190, 1.9997, 2.4064,
  

## Analysis Model Evaluation

In [18]:
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.598384,0.161196,13.000000,0.000000,65651.343750,131328.687500,1.314555e+05
GRU,0.612817,0.159032,6852.000000,0.000000,62604.199219,138912.406250,2.057781e+05
SPICE-RNN,0.592432,0.159869,102062.000000,0.000000,66929.343750,337982.687500,1.333962e+06
SPICE,0.551888,0.165688,12.105569,2.677257,75992.382812,152008.984375,1.521271e+05


## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )

## Generative Behavior Analysis

In [ ]:
estimator.eval()
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice.csv'
)

estimator.use_sindy(False)
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice_rnn.csv'
)

gru.eval()
generate_behavior(
    model=gru,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_gru.csv'
)

generate_behavior(
    model=cfs,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_cfs.csv'
)

In [ ]:
analysis_generative_behavior(
    path_data_real=path_data,
    path_data_gru='data/eckstein2024_gru.csv',
    path_data_benchmark='data/eckstein2024_cfs.csv',
    path_data_spice='data/eckstein2024_spice.csv',
    path_data_spice_rnn='data/eckstein2024_spice_rnn.csv',
    output_dir='results',
)